<a href="https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/notebooks/X0_raiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Números, movimento e Kalman

## De ℝ aos quaternions tridentes, em oito domínios e quatro etapas

**O que é esta série.** Uma sequência de notebooks em que a **mesma história** é contada oito vezes, cada vez num "domínio" de números mais rico: dos reais aos quaternions tridentes. Em cada domínio, quatro etapas:

- **(a) álgebra:** as operações, as propriedades que valem e as que se perdem, e a implementação em Python (inclusive com `sympy`);
- **(b) sistemas de equações** lineares e não lineares naquele domínio;
- **(c) cinemática e dinâmica:** as equações diferenciais do movimento, resolvidas simbolicamente pelo `sympy` e numericamente por Runge–Kutta (e, quando há rotações, por RKMK, o Runge–Kutta de Munthe-Kaas);
- **(d) perseguição:** um perseguidor estima, com um filtro de Kalman, onde está um alvo que só observa por sensores com ruído.

**Por que assim.** Quaternions, dual quaternions e quaternions tridentes assustam quando aparecem de uma vez. Aqui eles chegam aos poucos: cada domínio acrescenta **uma** ideia ao anterior (uma segunda direção, um infinitesimal, uma rotação, uma translação, uma velocidade), e as quatro etapas mostram o que essa ideia muda na conta, na equação, na simulação e no filtro.

**Para quem.** Alunos de Ciência da Computação. Supõe-se Python, NumPy, álgebra linear e probabilidade básicas. Não se supõe Física além de $F=ma$, nem Engenharia de Controle. Quem fez a série *O Filtro de Kalman para Cientistas da Computação* reconhecerá o Pepé e a Penélope, mas ela não é pré-requisito.

In [1]:
# nada a executar aqui além da verificação no final; esta célula só confirma o ambiente
import sys, numpy, matplotlib
print("Python", sys.version.split()[0], "| NumPy", numpy.__version__, "| Matplotlib", matplotlib.__version__)
try:
    import sympy; print("SymPy", sympy.__version__)
except ImportError:
    print("SymPy não encontrado: no Colab ele já vem instalado; localmente, use  pip install sympy")

Python 3.13.18 | NumPy 2.5.3 | Matplotlib 3.11.2


SymPy 1.14.0


---

## 1. O mapa: oito domínios × quatro etapas

A série é uma **árvore**: esta raiz (nível 0), um nó por domínio (nível I, que apresenta o domínio e os exemplos) e quatro folhas por domínio (nível II, as etapas a–d). Na prática, as folhas formam uma **matriz**, e ela pode ser lida de dois jeitos:
- **por linha** (um domínio inteiro, de (a) a (d)): é a ordem natural de uma primeira leitura;
- **por coluna** (uma etapa através dos domínios): por exemplo, a coluna (c) mostra o Runge–Kutta virando RKMK, e a coluna (d) mostra o filtro de Kalman virando o ESKF.

| domínio | nó do domínio | (a) álgebra | (b) sistemas de equações lineares e não lineares | (c) cinemática e dinâmica | (d) perseguição com filtro de Kalman |
|---|---|---|---|---|---|
| **I** · números reais ℝ | [X1_reais.ipynb](X1_reais.ipynb) | [a](X1a_reais_algebra.ipynb) · [Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/notebooks/X1a_reais_algebra.ipynb) | [b](X1b_reais_sistemas.ipynb) · [Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/notebooks/X1b_reais_sistemas.ipynb) | [c](X1c_reais_edo.ipynb) · [Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/notebooks/X1c_reais_edo.ipynb) | [d](X1d_reais_perseguicao.ipynb) · [Colab](https://colab.research.google.com/github/saulobortolon/Quaternios-Kalman-CC/blob/main/notebooks/X1d_reais_perseguicao.ipynb) |
| **II** · vetores de ℝ² | *em preparação* | — | — | — | — |
| **III** · números duais | *em preparação* | — | — | — | — |
| **IV** · números complexos ℂ | *em preparação* | — | — | — | — |
| **V** · números dual-complexos | *em preparação* | — | — | — | — |
| **VI** · quaternions | *em preparação* | — | — | — | — |
| **VII** · dual quaternions | *em preparação* | — | — | — | — |
| **VIII** · quaternions tridentes | *em preparação* | — | — | — | — |

---

## 2. A escada de domínios

| | números | grupo de movimentos | exemplo físico (b, c) | perseguição (d) |
|---|---|---|---|---|
| **I** | números reais ℝ | ℝ (translações na reta) | bloco com mola, atrito e força aplicada | ponto persegue ponto na reta, medindo uma função da distância |
| **II** | vetores de ℝ² | ℝ² (translações no plano) | projétil com arrasto | Pepé e Penélope no plano |
| **III** | números duais | SO(2)×ℝ (giro e deslizamento no mesmo eixo) | barra que gira e desliza num eixo | barra que se alinha a um alvo no eixo |
| **IV** | números complexos ℂ | ℂ (plano, com rotações por multiplicação) | pêndulo de Foucault | perseguição no plano em notação complexa |
| **V** | números dual-complexos | SE(2) (posição e atitude no plano) | barra ou carrinho no plano | veículo no plano que persegue e aponta |
| **VI** | quaternions | SO(3) (atitude no espaço) | corpo girando livre (efeito da raquete de tênis) | câmera que gira para manter o alvo centrado |
| **VII** | dual quaternions | SE(3) (pose no espaço) | paralelepípedo lançado girando | drone persegue drone |
| **VIII** | quaternions tridentes | SE₂(3) (pose e velocidade no espaço) | paralelepípedo ou drone com IMU | drone interceptador que persegue, aponta e lança uma rede |

**O que cada degrau acrescenta, e o que perde.**

| domínio | estrutura algébrica | acrescenta | perde em relação ao anterior |
|---|---|---|---|
| I · ℝ | corpo ordenado | — | — |
| II · ℝ² | espaço vetorial | uma segunda direção | o produto entre elementos e a ordem |
| III · duais $a+\varepsilon b$, $\varepsilon^2=0$ | anel comutativo | um infinitesimal: giro e deslizamento no mesmo eixo; derivação automática | inversos ($\varepsilon$ não tem inverso) |
| IV · ℂ | corpo | a rotação como multiplicação | a ordem |
| V · dual-complexos | anel não comutativo | posição **e** atitude no plano | a comutatividade |
| VI · ℍ | anel de divisão | rotações no espaço | (volta a ter inversos, mantém a não comutatividade) |
| VII · dual quaternions | anel não comutativo | pose no espaço (parafuso) | os inversos |
| VIII · quaternions tridentes | anel não comutativo | pose **e velocidade** no espaço | — |

O degrau VIII é o destino: um quaternion tridente é um quaternion cujos coeficientes são "números tridentes" $a+\varepsilon_1b+\varepsilon_2c$, com $\varepsilon_1^2=\varepsilon_2^2=\varepsilon_1\varepsilon_2=0$. Para quem chegou até ali pela escada, ele é um dual quaternion com um infinitesimal a mais.

---

## 3. A interface que atravessa a série

Em todos os domínios, o estado de um objeto em movimento vive num conjunto com seis operações: compor dois movimentos, inverter um movimento, a identidade, $\mathrm{Exp}$ (o movimento produzido por uma velocidade constante), $\mathrm{Log}$ (a velocidade que produz um movimento), e o par $\boxplus$, $\boxminus$ (somar e medir incrementos pequenos). A etapa (a) de cada domínio implementa a interface e roda **os mesmos testes**.

| domínio | $x\boxplus\delta$ | integrador da etapa (c) | filtro da etapa (d) |
|---|---|---|---|
| ℝ, ℝ² | $x+\delta$ | RK4 | KF, EKF, IEKF, filtro em grade |
| duais | $(\theta+\delta_\theta)+\varepsilon(d+\delta_d)$ | RK4 | EKF com jacobianos por números duais |
| ℂ | $z+\delta$ ou $z\,e^{i\delta}$ | RK4 = RKMK (comutativo) | EKF |
| dual-complexos | $\hat z\,\mathrm{Exp}(\delta)$ | RKMK | ESKF |
| quaternions | $q\otimes\mathrm{Exp}(\delta)$ | RKMK | ESKF |
| dual quaternions | $\hat q\otimes\mathrm{Exp}(\delta)$ | RKMK | ESKF |
| tridentes | $\tilde q\otimes\mathrm{Exp}(\delta)$ | RKMK | ESKF tridente |

---

## 4. Como usar

**No Colab, sem instalar nada.** Cada notebook tem, no topo, um botão "Abrir no Colab", e a matriz da seção 1 tem um link "Colab" em cada célula. Os notebooks são **autossuficientes**: cada um repete, numa célula inicial de ferramentas, o pouco que precisa das etapas anteriores, e roda sozinho.

**O repositório.** Todos os arquivos estão em [github.com/saulobortolon/Quaternios-Kalman-CC](https://github.com/saulobortolon/Quaternios-Kalman-CC):
- `notebooks/`: os notebooks, já executados (com figuras e resultados);
- `construtor/`: o código Python que **gera** os notebooks. Os notebooks não são editados à mão: cada correção é feita no construtor, e a série inteira é regenerada e reexecutada.

**Convenções**
- Toda afirmação numérica importante é conferida por uma célula que imprime `OK` (ou interrompe o notebook com uma mensagem clara).
- As simulações de Monte Carlo têm um parâmetro `N_MC` no topo do notebook: diminua-o para rodar mais rápido em aula.
- As leituras dos resultados ("**Leitura.**") foram escritas a partir dos números obtidos, e dizem o que os números dizem, inclusive quando contrariam a expectativa.

**A série irmã.** *O Filtro de Kalman para Cientistas da Computação* (guia, Pepé Le Gambá, IMM, notas de Im, quadrotor) está na [pasta do Google Drive](https://drive.google.com/drive/folders/1dtp9t1VQoMD-0hpIQoRVyFf0MT8C0M0f?usp=sharing).

---

## 5. Verificação: quais notebooks estão nesta pasta?

In [2]:
import os
from IPython.display import Markdown, display
SERIE = ['X0_raiz.ipynb', 'X1_reais.ipynb', 'X1a_reais_algebra.ipynb', 'X1b_reais_sistemas.ipynb', 'X1c_reais_edo.ipynb', 'X1d_reais_perseguicao.ipynb']
linhas = ["| notebook | nesta pasta? |", "|---|---|"]
for arq in SERIE:
    linhas.append(f"| {arq} | {'sim' if os.path.exists(arq) else '**não encontrado**'} |")
display(Markdown("\n".join(linhas)))
print(f"{sum(os.path.exists(a) for a in SERIE)} de {len(SERIE)} notebooks encontrados em {os.getcwd()}")

| notebook | nesta pasta? |
|---|---|
| X0_raiz.ipynb | sim |
| X1_reais.ipynb | sim |
| X1a_reais_algebra.ipynb | sim |
| X1b_reais_sistemas.ipynb | sim |
| X1c_reais_edo.ipynb | sim |
| X1d_reais_perseguicao.ipynb | sim |

6 de 6 notebooks encontrados em /home/claude/dominios/notebooks
